# Kata 06 — Errores Estructurados al estilo MCP

> Spec: [`specs/006-mcp-errors/spec.md`](../../specs/006-mcp-errors/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap
from katas._shared.eventlog import Logger
import json, time, random

client, settings = bootstrap(budget_calls=12)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Cuando una herramienta falla, NO devuelvo `"Error: failed"`. Devuelvo un payload tipado con `isError`, `errorCategory`, `isRetryable` y, si aplica, `retryAfterSeconds`. El cliente decide reintento, escalada o aborto desde flags, no desde prosa.

## 2. Por qué importa

Un string genérico fuerza al modelo a adivinar política de retry. Con flags tipados, esa decisión sale del modelo y entra en código auditable.

## 3. Ejemplo correcto — error tipado + cliente que reacciona

In [ ]:
def flaky_billing_lookup(args, attempt_state={"n": 0}):
    """Falla 2 veces con rate_limit, después funciona."""
    attempt_state["n"] += 1
    if attempt_state["n"] <= 2:
        return {
            "isError": True,
            "errorCategory": "rate_limit",
            "isRetryable": True,
            "retryAfterSeconds": 1,
            "explanation": "API quota momentánea",
        }
    return {"isError": False, "data": {"invoice_id": args["invoice_id"], "amount": 100.0}}

BILLING_TOOL = {
    "name": "billing_lookup",
    "description": "Consulta facturación. Puede devolver errores estructurados.",
    "input_schema": {
        "type": "object",
        "properties": {"invoice_id": {"type": "string"}},
        "required": ["invoice_id"],
    },
}

MAX_RETRIES = 3

def run_with_retry_logic(client, *, system, messages, tools, handler):
    log = Logger()
    iteration = 0
    retries = {}
    while True:
        iteration += 1
        resp = client.messages.create(system=system, messages=messages, tools=tools)
        if resp.stop_reason == "end_turn":
            log.add(iter=iteration, branch="halt", cause="end_turn")
            return resp, log
        if resp.stop_reason != "tool_use":
            log.add(iter=iteration, branch="halt", cause=f"unhandled:{resp.stop_reason}")
            return resp, log
        tu = next(b for b in resp.content if b.type == "tool_use")
        result = handler(tu.input)
        if result.get("isError"):
            cat = result["errorCategory"]
            if result.get("isRetryable") and retries.get(tu.name, 0) < MAX_RETRIES:
                retries[tu.name] = retries.get(tu.name, 0) + 1
                log.add(iter=iteration, tool=tu.name, error=cat, action="retry_client_side", attempt=retries[tu.name])
                time.sleep(result.get("retryAfterSeconds", 0))
                # Re-ejecutar el handler sin re-llamar al modelo
                continue_inner = True
                while continue_inner and retries[tu.name] < MAX_RETRIES:
                    result = handler(tu.input)
                    if not result.get("isError"):
                        continue_inner = False
                    else:
                        retries[tu.name] += 1
                        log.add(iter=iteration, tool=tu.name, error=result.get("errorCategory"), action="retry", attempt=retries[tu.name])
        messages.append({"role": "assistant", "content": resp.content})
        messages.append({"role": "user", "content": [{
            "type": "tool_result", "tool_use_id": tu.id, "content": json.dumps(result),
        }]})
        log.add(iter=iteration, tool=tu.name, final_status="ok" if not result.get("isError") else result.get("errorCategory"))

In [ ]:
system = "Eres un asistente. Si el usuario pregunta por una factura, llama a billing_lookup."
messages = [{"role": "user", "content": "Status de la factura INV-77?"}]
final, log = run_with_retry_logic(client, system=system, messages=messages, tools=[BILLING_TOOL], handler=flaky_billing_lookup)
log.show()
print("\nrespuesta final:", next((b.text for b in final.content if b.type == "text"), ""))

El retry vivió en el cliente, totalmente fuera del modelo. El modelo sólo vio el `tool_result` exitoso al final.

## 4. Anti-patrón — error como string

In [ ]:
def flaky_string(args, attempt_state={"n": 0}):
    attempt_state["n"] += 1
    if attempt_state["n"] <= 2:
        # Simula la peor versión: lanza excepción que se convierte en string crudo
        return f"Error: something went wrong (attempt {attempt_state['n']})"
    return json.dumps({"invoice_id": args["invoice_id"], "amount": 100.0})

def run_dumb(client, *, system, messages, tools, handler):
    log = Logger()
    for i in range(5):
        resp = client.messages.create(system=system, messages=messages, tools=tools)
        if resp.stop_reason == "end_turn":
            log.add(iter=i+1, branch="halt")
            return resp, log
        if resp.stop_reason != "tool_use":
            return resp, log
        tu = next(b for b in resp.content if b.type == "tool_use")
        result = handler(tu.input)
        log.add(iter=i+1, tool=tu.name, raw_result=str(result)[:60])
        messages.append({"role": "assistant", "content": resp.content})
        messages.append({"role": "user", "content": [{
            "type": "tool_result", "tool_use_id": tu.id, "content": str(result),
        }]})
    return resp, log

print("=== sin retry estructurado ===")
final_bad, log_bad = run_dumb(client, system=system, messages=[{"role":"user","content":"Status INV-88?"}], tools=[BILLING_TOOL], handler=flaky_string)
log_bad.show()
print("\nrespuesta:", next((b.text for b in final_bad.content if b.type == "text"), "")[:300])

Sin metadata, el modelo termina razonando sobre el string `"Error: something went wrong"`. Puede:

- abandonar y disculparse,
- intentar de nuevo gastando turnos,
- inventar un workaround.

Cualquiera de las tres es un fallo.

## 5. Argumento de certificación

- **Tres ejes de error**: `isError`, `errorCategory`, `isRetryable`. Cubren las decisiones del cliente sin que el modelo razone sobre prosa.
- **Política de retry vive en el cliente.** El modelo no debe decidir cuántas veces reintentar.
- **`explanation` es para el humano que audita el log**, no para que el modelo lo parsee.
- **Conexión con otros katas**: errores no recuperables (auth, validation) escalan al humano (Kata 16); errores transitorios respetan rate-limit (Kata 17 si el batch falla parcialmente).

## 6. Auto-evaluación

**1. ¿Cómo trato un error que llega sin `errorCategory`?**

Lo trato como `errorCategory="unknown"` y `isRetryable=false` por defecto. Loguear y escalar es preferible a un retry ciego. La defensa es estricta: el cliente sólo confía en lo que el contrato declara.

**2. ¿Cuál es la diferencia entre `transient` y `rate_limit` en política de retry?**

`transient` es backoff exponencial corto (ms-segundos), `rate_limit` respeta `retryAfterSeconds` que viene del proveedor. Son curvas de espera distintas; la categoría se la doy al ejecutor de retry, no al modelo.

**3. ¿Qué prueba reintroduce el anti-patrón (string genérico) y qué assert falla?**

Llamar al handler `flaky_string` con el bucle robusto: el bucle no encuentra `isError`/`isRetryable`, asume éxito, y el modelo recibe un `tool_result` falso. Una aserción defensiva: validar el shape del result con un schema antes de pasarlo al historial; falla si llega un string crudo.